# 🎙️ microWakeWord Training — Kokoro TTS + GPU Moderna

Pipeline de entrenamiento para un modelo wake word compatible con **ESP32-S3** (via ESPHome/microWakeWord).  
Genera muestras de voz sintéticas con **Kokoro TTS** usando múltiples voces en español para mayor diversidad.

### Requisitos del sistema
- Python 3.10 o 3.11
- GPU NVIDIA moderna (RTX 3000+ recomendada)
- CUDA 12.x + cuDNN 8.x
- ~20 GB de espacio libre en disco

### Flujo general
1. Instalar dependencias
2. Configurar GPU
3. Generar muestras con Kokoro TTS (múltiples voces/tonos)
4. Descargar datasets negativos
5. Augmentar y generar espectrogramas
6. Entrenar el modelo
7. Exportar `.tflite` para ESP32-S3

---
## 📦 Celda 1 — Instalación de dependencias
Ejecutar una sola vez. Reiniciar el kernel después si usas Jupyter clásico.

In [1]:
import platform, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

# --- microWakeWord ---
if platform.system() == 'Darwin':
    pip('git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version')

pip('git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f')

import os
if not os.path.exists('microWakeWord'):
    subprocess.check_call(['git', 'clone', 'https://github.com/kahrendt/microWakeWord'])

pip('-e', './microWakeWord')

# --- Kokoro TTS ---
pip('kokoro>=0.9.4', 'soundfile', 'numpy')

# --- Herramientas de audio y training ---
pip('datasets', 'scipy', 'tqdm', 'tensorboard', 'audiomentations', 'mmap_ninja')

print('✅ Instalación completa. Reinicia el kernel si es la primera vez.')


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
piper-sample-generator 3.2.0 requires audiomentations==0.33.0, but you have audiomentations 0.43.1 which is incompatible.
piper-sample-generator 3.2.0 requires numpy<3,>=2, but you have numpy 1.26.4 which is incompatible.
tts 0.22.0 requires pandas<2.0,>=1.4, but you have pandas 3.0.2 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


✅ Instalación completa. Reinicia el kernel si es la primera vez.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import subprocess, os

# Encontrar librerías CUDA del sistema
def find_lib(name):
    result = subprocess.run(['find', '/usr', '/opt', '-name', name, '-type', 'f'], 
                          capture_output=True, text=True)
    paths = [p for p in result.stdout.strip().split('\n') if p]
    return paths

# Forzar CUDA 12 desde el venv de pip (nvidia-* packages)
import sys
venv_base = os.path.join(sys.prefix, 'lib', 
                         f'python{sys.version_info.major}.{sys.version_info.minor}',
                         'site-packages', 'nvidia')

lib_dirs = []
for pkg in ['cuda_runtime', 'cudnn', 'cublas', 'cufft', 'curand', 
            'cusolver', 'cusparse', 'nccl', 'cuda_nvrtc']:
    path = os.path.join(venv_base, pkg, 'lib')
    if os.path.isdir(path):
        lib_dirs.append(path)
        print(f'✅ Encontrado: {path}')
    else:
        print(f'❌ Faltante:   {path}')

os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=' + os.path.join(sys.prefix, 'lib', 
    f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages', 'nvidia', 'cuda_runtime')

print(f'\n📚 LD_LIBRARY_PATH configurado con {len(lib_dirs)} rutas')

✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cudnn/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cublas/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cufft/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/curand/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cusolver/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cusparse/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/nccl/lib
✅ Encontrado: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/cuda_nvrtc/lib

📚 LD_LIBRARY_PATH configurado con 9 rutas


In [4]:
# Evitar que TF use el CUDA del sistema en /opt/cuda
export LD_LIBRARY_PATH=$(python -c "
import sys, os, glob
base = os.path.join(sys.prefix, 'lib/python3.11/site-packages/nvidia')
dirs = []
for d in glob.glob(base + '/*/lib'):
    if 'cu13' not in d:
        dirs.append(d)
print(':'.join(dirs))
")

python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"

SyntaxError: unterminated string literal (detected at line 2) (4230766744.py, line 2)

---
## ⚡ Celda 2 — Configuración de GPU
Detecta y configura la GPU. Compatible con RTX 3000/4000 series, A100, etc.

In [2]:
import os, sys, ctypes, glob

# ── Apuntar las librerías NVIDIA del venv ──────────────────────────────────────
venv_nvidia_base = os.path.join(sys.prefix, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages', 'nvidia')
lib_paths = []
for pkg in ['cudnn', 'nccl', 'cublas', 'cufft', 'curand', 'cusolver', 'cusparse']:
    candidate = os.path.join(venv_nvidia_base, pkg, 'lib')
    if os.path.isdir(candidate):
        lib_paths.append(candidate)

system_cuda = '/usr/local/cuda/lib64'
if os.path.isdir(system_cuda):
    lib_paths.append(system_cuda)

existing = os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_paths) + (':' + existing if existing else '')

# ── Precargar NCCL para evitar el error de símbolo ncclCommWindowDeregister ───
nccl_libs = glob.glob(os.path.join(venv_nvidia_base, 'nccl', 'lib', 'libnccl.so*'))
if nccl_libs:
    try:
        ctypes.CDLL(sorted(nccl_libs)[0], mode=ctypes.RTLD_GLOBAL)
        print(f'🔗 NCCL precargada: {sorted(nccl_libs)[0]}')
    except Exception as e:
        print(f'⚠️  NCCL no pudo precargarse: {e}')

# ── Configuración de TensorFlow ────────────────────────────────────────────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'          # Silenciar logs verbosos
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'  # Mejor para GPUs modernas
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # Memoria dinámica
os.environ['HF_HUB_OFFLINE'] = '0'                # Necesario para descargar Kokoro

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'🚀 GPU detectada: {[g.name for g in gpus]}')
    print(f'   TensorFlow: {tf.__version__}')
else:
    print('⚠️  No se detectó GPU. El entrenamiento será muy lento en CPU.')
    print('   Verifica que CUDA y cuDNN estén correctamente instalados.')

🔗 NCCL precargada: /home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/nvidia/nccl/lib/libnccl.so.2
🚀 GPU detectada: ['/physical_device:GPU:0']
   TensorFlow: 2.21.0


---
## 🗣️ Celda 3 — Generación de muestras con Kokoro TTS

Genera ~1000+ muestras de la wake word usando **múltiples voces en español** de Kokoro.  
La diversidad de voces es crítica para que el modelo sea robusto en el mundo real.

**Voces disponibles en español (Kokoro):**
- `ef_dora` — femenina, acento neutro latinoamericano
- `em_alex` — masculina, acento neutro latinoamericano  
- `em_santa` — masculina, tono más grave
- `ef_bella` — femenina, tono más suave (puede ser usada con `lang_code='e'`)

Ajusta `TARGET_WORD` con tu wake word exacta.

In [3]:
import os
import random
import numpy as np
import soundfile as sf
from tqdm import tqdm
from kokoro import KPipeline

# ══════════════════════════════════════════════════
#  ⚙️  CONFIGURACIÓN — Ajusta estos valores
# ══════════════════════════════════════════════════
TARGET_WORD    = "jey ardo"   # Tu wake word exacta
OUTPUT_DIR     = "generated_samples"
SAMPLES_TOTAL  = 1200          # Total de muestras a generar
KOKORO_DEVICE  = 'cuda'        # 'cuda' para GPU, 'cpu' para fallback

# Voces a usar y su peso relativo en la distribución de muestras
# Más voces = mejor robustez del modelo
VOICE_CONFIG = [
    # (nombre_voz,   peso,  variaciones_de_velocidad)
    ('em_alex',      0.30,  (0.85, 1.15)),  # Masculina principal (México)
    ('ef_dora',      0.30,  (0.85, 1.15)),  # Femenina principal  (México)
    ('em_santa',     0.20,  (0.80, 1.10)),  # Masculina grave
]
# ══════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Calcular cuántas muestras por voz
voice_counts = {}
total_weight = sum(w for _, w, _ in VOICE_CONFIG)
remaining = SAMPLES_TOTAL
for i, (voice, weight, _) in enumerate(VOICE_CONFIG):
    if i < len(VOICE_CONFIG) - 1:
        count = round(SAMPLES_TOTAL * weight / total_weight)
        remaining -= count
    else:
        count = remaining  # Asignar sobrante a la última voz
    voice_counts[voice] = count

print(f"🎙️  Wake word: '{TARGET_WORD}'")
print(f"📊  Distribución de muestras:")
for voice, _, _ in VOICE_CONFIG:
    print(f"    • {voice}: {voice_counts[voice]} muestras")
print(f"    TOTAL: {SAMPLES_TOTAL} muestras\n")

sample_idx = 0

for voice_name, weight, speed_range in VOICE_CONFIG:
    n_samples = voice_counts[voice_name]
    print(f"⏳ Generando {n_samples} muestras con voz '{voice_name}'...")

    # Cargar pipeline para esta voz
    try:
        pipeline = KPipeline(lang_code='e', device=KOKORO_DEVICE)
    except Exception:
        print(f"  ⚠️  No se pudo cargar en {KOKORO_DEVICE}, usando CPU.")
        pipeline = KPipeline(lang_code='e', device='cpu')

    with tqdm(total=n_samples, desc=f"  {voice_name}") as pbar:
        for i in range(n_samples):
            speed = round(random.uniform(*speed_range), 3)
            out_path = os.path.join(OUTPUT_DIR, f"{sample_idx:05d}.wav")

            try:
                generator = pipeline(TARGET_WORD, voice=voice_name, speed=speed)
                chunks = [audio for _, _, audio in generator]

                if chunks:
                    audio_full = np.concatenate(chunks)
                    # Kokoro genera a 24kHz — microWakeWord necesita 16kHz
                    # Resamplear con scipy
                    from scipy.signal import resample_poly
                    audio_16k = resample_poly(audio_full, up=2, down=3)  # 24000 * 2/3 = 16000
                    gain = round(random.uniform(0.1, 1.0), 3)
                    audio_16k = audio_16k * gain
                    audio_16k = np.clip(audio_16k, -1.0, 1.0)
                    audio_16k_int = (audio_16k * 32767).astype(np.int16)
                    sf.write(out_path, audio_16k_int, 16000)
                    sample_idx += 1
                else:
                    print(f"  ⚠️  Kokoro no generó audio para muestra {sample_idx}")

            except Exception as e:
                print(f"  ❌ Error en muestra {sample_idx}: {e}")

            pbar.update(1)

    del pipeline  # Liberar VRAM entre voces

print(f"\n✅ Generación completa: {sample_idx} muestras guardadas en '{OUTPUT_DIR}/'")

/home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🎙️  Wake word: 'jey ardo'
📊  Distribución de muestras:
    • em_alex: 450 muestras
    • ef_dora: 450 muestras
    • em_santa: 300 muestras
    TOTAL: 1200 muestras

⏳ Generando 450 muestras con voz 'em_alex'...


/home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
  em_alex: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 450/450 [00:22<00:00, 20.22it/s]


⏳ Generando 450 muestras con voz 'ef_dora'...


/home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/sesgaro/microWakeWord/venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
  ef_dora: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 450/450 [00:20<00:00, 21.54it/s]


⏳ Generando 300 muestras con voz 'em_santa'...


  em_santa: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:13<00:00, 21.51it/s]


✅ Generación completa: 1200 muestras guardadas en 'generated_samples/'


---
## 🔊 Celda 4 — Verificar una muestra generada
Reproduce una muestra al azar para verificar la calidad del audio.

In [12]:
import random, glob
from IPython.display import Audio, display

samples = glob.glob(f'{OUTPUT_DIR}/*.wav')
sample = random.choice(samples)
print(f'🎧 Reproduciendo: {sample}')
display(Audio(sample, autoplay=True))

🎧 Reproduciendo: generated_samples/00985.wav


---
## 📥 Celda 5 — Descargar datasets negativos
Los datasets negativos son audios que el modelo debe aprender a **ignorar** (conversaciones, música, ruido).  
Esto es igual de importante que el dataset positivo.

In [13]:
import os

# ── Dataset negativos pre-generados de microWakeWord ──────────────────────────
NEG_DIR = './negative_datasets'
if not os.path.exists(NEG_DIR):
    os.makedirs(NEG_DIR, exist_ok=True)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        zip_path = f'{NEG_DIR}/{fname}'
        print(f'⏳ Descargando {fname}...')
        os.system(f'wget -q --show-progress -O {zip_path} {link_root + fname}')
        os.system(f'unzip -q {zip_path} -d {NEG_DIR}')
        os.remove(zip_path)
    print('✅ Datasets negativos descargados.')
else:
    print('✅ Datasets negativos ya existen. Saltando descarga.')

✅ Datasets negativos ya existen. Saltando descarga.


---
## 🌊 Celda 6 — Descargar audios de fondo para augmentación
Ruidos de fondo reales hacen que el modelo sea más robusto en entornos ruidosos.

In [14]:
import datasets as hf_datasets
import scipy.io.wavfile
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm

# ── Respuestas de impulso (Room Impulse Responses) ────────────────────────────
RIR_DIR = './mit_rirs'
if not os.path.exists(RIR_DIR):
    print('⏳ Descargando MIT Room Impulse Responses...')
    os.makedirs(RIR_DIR, exist_ok=True)
    rir_dataset = hf_datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    for row in tqdm(rir_dataset, desc='Procesando RIR'):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            os.path.join(RIR_DIR, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print('✅ RIRs descargadas.')
else:
    print('✅ MIT RIRs ya existen.')

# ── AudioSet (ruidos y ambiente) ──────────────────────────────────────────────
AUDIOSET_16K_DIR = './audioset_16k'
if not os.path.exists(AUDIOSET_16K_DIR):
    AUDIOSET_RAW_DIR = './audioset'
    if not os.path.exists(AUDIOSET_RAW_DIR):
        print('⏳ Descargando AudioSet...')
        os.makedirs(AUDIOSET_RAW_DIR, exist_ok=True)
        link = 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar'
        os.system(f'wget -q --show-progress -O {AUDIOSET_RAW_DIR}/bal_train09.tar {link}')
        os.system(f'cd {AUDIOSET_RAW_DIR} && tar -xf bal_train09.tar')

    print('⚙️  Convirtiendo AudioSet a 16kHz...')
    os.makedirs(AUDIOSET_16K_DIR, exist_ok=True)
    flac_files = list(Path(f'{AUDIOSET_RAW_DIR}/audio').glob('**/*.flac'))
    ds = hf_datasets.Dataset.from_dict({'audio': [str(f) for f in flac_files]})
    ds = ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc='AudioSet → 16kHz'):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(
            os.path.join(AUDIOSET_16K_DIR, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print('✅ AudioSet 16kHz listo.')
else:
    print('✅ AudioSet 16kHz ya existe.')

# ── Free Music Archive (música de fondo) ──────────────────────────────────────
FMA_16K_DIR = './fma_16k'
if not os.path.exists(FMA_16K_DIR):
    FMA_RAW_DIR = './fma'
    if not os.path.exists(FMA_RAW_DIR):
        print('⏳ Descargando Free Music Archive (FMA xsmall)...')
        os.makedirs(FMA_RAW_DIR, exist_ok=True)
        link = 'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip'
        os.system(f'wget -q --show-progress -O {FMA_RAW_DIR}/fma_xs.zip {link}')
        os.system(f'cd {FMA_RAW_DIR} && unzip -q fma_xs.zip')

    print('⚙️  Convirtiendo FMA a 16kHz...')
    os.makedirs(FMA_16K_DIR, exist_ok=True)
    mp3_files = list(Path(f'{FMA_RAW_DIR}/fma_small').glob('**/*.mp3'))
    ds = hf_datasets.Dataset.from_dict({'audio': [str(f) for f in mp3_files]})
    ds = ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc='FMA → 16kHz'):
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(
            os.path.join(FMA_16K_DIR, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print('✅ FMA 16kHz listo.')
else:
    print('✅ FMA 16kHz ya existe.')

print('\n✅ Todos los audios de fondo están listos.')

✅ MIT RIRs ya existen.
✅ AudioSet 16kHz ya existe.
✅ FMA 16kHz ya existe.

✅ Todos los audios de fondo están listos.


---
## 🔬 Celda 7 — Configurar augmentaciones y clips
La augmentación simula condiciones del mundo real: ruido, reverberación, cambios de volumen, etc.

In [15]:
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# ── Cargar clips generados por Kokoro ─────────────────────────────────────────
clips = Clips(
    input_directory=OUTPUT_DIR,
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=42,
    split_count=0.1,   # 10% para validación/test
)
import glob
n = len(glob.glob(f'{OUTPUT_DIR}/*.wav'))
print(f'📂 Clips cargados: {n} archivos en {OUTPUT_DIR}/')

# ── Definir augmentaciones ────────────────────────────────────────────────────
# Estos valores están pensados para simular el entorno de un ESP32-S3
# con micrófono MEMS integrado a distancias de 0.5–3 metros
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15,    # Simula diferentes respuestas de micrófono
        'TanhDistortion':        0.10,    # Saturación ligera (micrófono MEMS barato)
        'PitchShift':            0.15,    # Variación de tono adicional
        'BandStopFilter':        0.10,    # Simula interferencias de RF
        'AddColorNoise':         0.20,    # Ruido de fondo electrónico
        'AddBackgroundNoise':    0.80,    # Ruido de ambiente real (crítico)
        'Gain':                  1.00,    # Siempre normalizar volumen
        'RIR':                   0.60,    # Reverberación de sala (muy importante)
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,     # Puede ser más ruidoso que la voz en el peor caso
    background_max_snr_db=15,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

print('✅ Augmentaciones configuradas.')

📂 Clips cargados: 1200 archivos en generated_samples/
✅ Augmentaciones configuradas.


---
## 🎧 Celda 8 — Verificar augmentación

In [18]:
from IPython.display import Audio, display
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented   = augmenter.augment_clip(random_clip)
save_clip(augmented, 'augmented_clip.wav')

print('🎧 Clip augmentado (con ruido de fondo y reverb):')
display(Audio('augmented_clip.wav', autoplay=True))

🎧 Clip augmentado (con ruido de fondo y reverb):


---
## 🗺️ Celda 9 — Generar espectrogramas para training/validation/testing
Convierte los clips de audio en espectrogramas (la representación que usa el modelo).  
Esta celda puede tardar varios minutos dependiendo del número de muestras.

In [19]:
import os, shutil
from mmap_ninja.ragged import RaggedMmap

FEATURES_DIR = 'generated_augmented_features'
os.makedirs(FEATURES_DIR, exist_ok=True)

SPLITS = {
    'training':   {'split_name': 'train',      'repetition': 2,  'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1,  'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1,  'slide_frames': 1},
}

for split, cfg in SPLITS.items():
    out_dir = os.path.join(FEATURES_DIR, split)
    mmap_dir = os.path.join(out_dir, 'wakeword_mmap')

    # Siempre borrar y regenerar
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
        print(f'🗑️  {split}: limpiado.')
    os.makedirs(out_dir, exist_ok=True)

    print(f'⏳ Generando espectrogramas para: {split}...')
    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=cfg['slide_frames'],
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=mmap_dir,
        sample_generator=spectrograms.spectrogram_generator(
            split=cfg['split_name'],
            repeat=cfg['repetition']
        ),
        batch_size=100,
        verbose=True,
    )
    print(f'  ✅ {split} listo.')

print('\n✅ Todos los espectrogramas generados.')

🗑️  training: limpiado.
⏳ Generando espectrogramas para: training...


19200it [00:14, 1313.14it/s]


  ✅ training listo.
🗑️  validation: limpiado.
⏳ Generando espectrogramas para: validation...


1200it [00:00, 1204.37it/s]


  ✅ validation listo.
🗑️  testing: limpiado.
⏳ Generando espectrogramas para: testing...


120it [00:00, 143.38it/s]

  ✅ testing listo.

✅ Todos los espectrogramas generados.


---
## ⚙️ Celda 10 — Generar configuración de entrenamiento (YAML)
Ajusta `training_steps` y los pesos de penalización para optimizar el modelo.  
Para la ESP32-S3 se recomienda el modelo `mixednet` que ya está optimizado para TFLite cuantizado.

In [20]:
import yaml

# ══════════════════════════════════════════════════
#  ⚙️  Parámetros de entrenamiento
# ══════════════════════════════════════════════════
TRAINING_STEPS        = [20000]   # Más pasos = mejor modelo (pero más tiempo)
POSITIVE_CLASS_WEIGHT = [1]       # Peso de muestras positivas (wake word)
NEGATIVE_CLASS_WEIGHT = [10]      # Peso de muestras negativas — sube para menos falsos positivos
LEARNING_RATE         = [0.001]
# ══════════════════════════════════════════════════

config = {
    'window_step_ms': 10,
    "window_step_ms": 10,
    "clip_duration_ms": 1000   , # Duración del clip en ms — estándar para wake words
    "average_window_duration_ms": 100,
    "detection_threshold": 0.5,
    "suppression_ms": 500,
    "minimum_count": 3,
    "batch_size": 128,
    "training_input_shape": [48, 40],
    "eval_step_interval": 500,
    "save_step_interval": 500,
    "minimization_metric": "ambient_false_positives_per_hour",
    "maximization_metric": "average_viable_recall",
    "target_minimization": 1.0,  # objetivo: menos de 1 falso positivo por hora
    "primary_metric": "accuracy",
    'train_dir': 'trained_models/wakeword',
    'time_mask_max_size': [5],
    'time_mask_count': [2],
    'freq_mask_max_size': [5],
    'freq_mask_count': [2],

    'features': [
        # ── Positivos (wake word generada con Kokoro) ──
        {
            'features_dir': 'generated_augmented_features',
            'sampling_weight': 2.0,
            'penalty_weight': 1.0,
            'truth': True,
            'truncation_strategy': 'truncate_start',
            'type': 'mmap',
        },
        # ── Negativos: speech (el más importante — evita confundir palabras parecidas) ──
        {
            'features_dir': 'negative_datasets/speech',
            'sampling_weight': 10.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        # ── Negativos: conversaciones en grupo ──
        {
            'features_dir': 'negative_datasets/dinner_party',
            'sampling_weight': 10.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        # ── Negativos: silencio / no-speech ──
        {
            'features_dir': 'negative_datasets/no_speech',
            'sampling_weight': 5.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        # ── Solo para evaluación (no se usa en training) ──
        {
            'features_dir': 'negative_datasets/dinner_party_eval',
            'sampling_weight': 0.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'split',
            'type': 'mma`p',
        },
    ],

    'training_steps':        TRAINING_STEPS,
    'positive_class_weight': POSITIVE_CLASS_WEIGHT,
    'negative_class_weight': NEGATIVE_CLASS_WEIGHT,
    'learning_rates':        LEARNING_RATE,
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('✅ training_parameters.yaml guardado.')
print(f'   Pasos: {TRAINING_STEPS[0]:,}')
print(f'   Peso negativo: {NEGATIVE_CLASS_WEIGHT[0]}  (sube para menos falsos positivos)')

✅ training_parameters.yaml guardado.
   Pasos: 20,000
   Peso negativo: 10  (sube para menos falsos positivos)


---
## 🏋️ Celda 11 — Entrenamiento

Entrena el modelo `mixednet` optimizado para ESP32-S3.  
La GPU se utilizará automáticamente si está disponible.

**Arquitectura:** MixedNet con convoluciones mixtas — diseñado para ser pequeño (~50K MACs) y correr en microcontroladores.

Al final del entrenamiento verás métricas como:
- `frr` = False Rejection Rate (la wake word se ignora) — debe ser < 0.05
- `faph` = False Accepts Per Hour (activa sin que digas la palabra) — debe ser < 1.0 idealmente

In [21]:
# Verificar GPU una vez más antes de entrenar
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'🖥️  Dispositivos disponibles: GPU={len(gpus)}, CPU disponible')
if gpus:
    print(f'   GPU: {gpus[0].name}')

🖥️  Dispositivos disponibles: GPU=1, CPU disponible
   GPU: /physical_device:GPU:0


In [25]:
# import subprocess, os, sys

env = os.environ.copy()
env['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
env['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
env['TF_CUDA_MALLOC_ASYNC_SUPPORTED_PREEMPTIVE_FREE_FRACTION'] = '0.5'
env['PYTHONIOENCODING'] = 'utf-8'
env['LANG'] = 'en_US.UTF-8'
cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config=training_parameters.yaml',
    '--train', '1',
    '--restore_checkpoint', '0',
    '--test_tf_nonstreaming', '0',
    '--test_tflite_nonstreaming', '0',
    '--test_tflite_nonstreaming_quantized', '0',
    '--test_tflite_streaming', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1,1,1,1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '2',
]

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, encoding='utf-8')
for line in process.stdout:
    print(line, end='')
process.wait()

INFO:absl:Loading and analyzing data sets.
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 153, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (128, 153, 1, 40) │          0 │ input_layer[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream (Stream)     │ (128, 75, 1, 32)  │      6,400 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (128, 75, 1, 32)  │          0 │ stream[0][0]      │
│ (Activation

0

---
## 📊 Celda 12 — Ver métricas con TensorBoard (opcional)

In [26]:
%load_ext tensorboard
%tensorboard --logdir trained_models/wakeword

---
## 📦 Celda 13 — Exportar modelo para ESP32-S3

El archivo `.tflite` es el modelo final cuantizado en INT8 que corre directamente en la ESP32-S3.

Para usarlo con ESPHome necesitas crear un JSON de manifiesto. Consulta:
- [Documentación ESPHome microWakeWord](https://esphome.io/components/micro_wake_word)
- [Ejemplos de modelos v2](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2)

Ajusta el umbral de probabilidad (`probability_cutoff`) según las métricas de la celda anterior:
- Un cutoff alto → menos falsos positivos pero más rechazos
- Un cutoff bajo → más sensible pero más falsos positivos

In [ ]:
import os, shutil

TFLITE_SOURCE = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
TFLITE_DEST   = f'{TARGET_WORD.replace(" ", "_")}_esp32s3.tflite'

if os.path.exists(TFLITE_SOURCE):
    shutil.copy(TFLITE_SOURCE, TFLITE_DEST)
    size_kb = os.path.getsize(TFLITE_DEST) / 1024
    print(f'✅ Modelo exportado: {TFLITE_DEST}')
    print(f'   Tamaño: {size_kb:.1f} KB')
    print(f'''
📋 Ejemplo de manifiesto ESPHome (model_manifest.json):
{{
  "type": "micro_wake_word_model",
  "wake_word": "{TARGET_WORD}",
  "author": "Tu nombre",
  "website": "",
  "version": "1",
  "micro": {{
    "model": "stream_state_internal_quant.tflite",
    "probability_cutoff": 0.88,
    "sliding_window_size": 5,
    "tensor_arena_size": 40000
  }}
}}
    ''')
else:
    print('❌ No se encontró el archivo .tflite. ¿El entrenamiento finalizó correctamente?')
    print(f'   Buscado en: {TFLITE_SOURCE}')

In [ ]:
# ── Descarga directa (si estás en Jupyter local) ───────────────────────────────
# En Colab: from google.colab import files; files.download(TFLITE_DEST)
# En local: el archivo ya está en el directorio de trabajo

print(f'📥 El archivo está en: {os.path.abspath(TFLITE_DEST)}')